<a href="https://www.kaggle.com/code/rawanmoamed/rawan-cnn?scriptVersionId=282662774" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install torch_scatter torcheeg 


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of mne to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of wfdb to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torcheeg.datasets import SEEDIVDataset
from torcheeg import transforms
import scipy.signal as signal
import random
import numpy as np

In [3]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

In [5]:
def BandPassFilter(eeg_data):
    b, a = signal.butter(4, Wn=[1.0, 75.0], btype='bandpass', fs=200)
    return signal.filtfilt(b, a, eeg_data, axis=-1)


In [6]:
def Notch(eeg_data):
    b, a = signal.iirnotch(w0=50.0, Q=30.0, fs=200)
    return signal.filtfilt(b, a, eeg_data, axis=-1)

In [7]:
# 2. Define Preprocessing
t_transform = transforms.Compose([
    transforms.Lambda(BandPassFilter),
    transforms.Lambda(Notch),
    transforms.BaselineRemoval(),
    transforms.MeanStdNormalize(),
    transforms.To2d()
    
])

In [8]:
# 3. Load Data
dataset = SEEDIVDataset(
    io_path='./tmp_out/seed_iv',
    root_path='/kaggle/input/seed-iv/eeg_raw_data',
    offline_transform=t_transform,
    label_transform=transforms.Compose([
        transforms.Select('emotion'),
    ]),
    chunk_size=800,  # 4 seconds
    num_worker=4
)

[2025-11-29 15:19:27] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv.
[2025-11-29 15:19:27] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 1it [00:03,  3.88s/it]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 8it [00:03,  2.73it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 15it [00:04,  5.95it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 22it [00:04,  9.97it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 28it [00:04, 14.08it/s]
[RECORD /kaggle/input/seed-iv/eeg_raw_data/1/4_20151111.mat]: 34it [00:04, 18.84it/s]
[RECORD /kaggle/input/s

In [9]:
import pandas as pd

# 1. Get the metadata DataFrame
df = dataset.info

# 2. Count the segments for each emotion
# 0: Neutral, 1: Sad, 2: Fear, 3: Happy
counts = df['emotion'].value_counts().sort_index()
total = len(df)

print(f"Total Segments: {total}")
print("-" * 30)
print("Count per Emotion:")
print(counts)

print("-" * 30)
print("Percentage per Emotion:")
percentages = (counts / total) * 100
print(percentages.round(2))

# 3. Check for Imbalance
# If the difference between max and min is > 10%, we might need a WeightedSampler
max_pct = percentages.max()
min_pct = percentages.min()

if (max_pct - min_pct) > 10:
    print(f"\n⚠️ WARNING: Data is IMBALANCED (Diff: {max_pct - min_pct:.2f}%)")
    print("Consider using a WeightedRandomSampler.")
else:
    print(f"\n✅ Data is reasonably BALANCED (Diff: {max_pct - min_pct:.2f}%)")

Total Segments: 37575
------------------------------
Count per Emotion:
emotion
0    10170
1    10245
2     9225
3     7935
Name: count, dtype: int64
------------------------------
Percentage per Emotion:
emotion
0    27.07
1    27.27
2    24.55
3    21.12
Name: count, dtype: float64

✅ Data is reasonably BALANCED (Diff: 6.15%)


In [10]:
import pandas as pd

min_count = df['emotion'].value_counts().min()

balanced_df = (
    df.groupby('emotion')
      .apply(lambda x: x.sample(min_count, replace=False, random_state=42))
      .reset_index(drop=True)
)

print(balanced_df['emotion'].value_counts())


emotion
0    7935
1    7935
2    7935
3    7935
Name: count, dtype: int64


/tmp/ipykernel_20/2625966507.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_count, replace=False, random_state=42))


In [11]:
from torch.utils.data import random_split


train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, test_set = random_split(dataset, [train_size, test_size])
print(f"Train samples: {len(train_set)}, Test samples: {len(test_set)}")



Train samples: 30060, Test samples: 7515


In [12]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch
train_indices=train_set.indices
train_labels = [df.iloc[i]['emotion'] for i in train_indices]
class_counts = pd.Series(train_labels).value_counts().to_dict()
weights = [1.0 / class_counts[label] for label in train_labels]

sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=64, sampler=sampler)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)


In [13]:
import torch.nn as nn



input_tensor = dataset[0][0]
n_channels = input_tensor.shape[1]  
samples = input_tensor.shape[2]    


#n_channels = dataset[0][0].shape[0]   
#samples = dataset[0][0].shape[1]
n_classes = 4                          
print(f"Electrodes (Height): {n_channels}, Time (Width): {samples}")
model = nn.Sequential(

    nn.Conv2d(1, 16, kernel_size=(1, 64), padding=(0, 32), bias=False),
    nn.BatchNorm2d(16),
    

    nn.Conv2d(16, 32, kernel_size=(62, 1), bias=False),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(1, 4)), 
    nn.Dropout(0.5),


    nn.Conv2d(32, 64, kernel_size=(1, 16), padding=(0, 8), bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(1, 4)), 
    nn.Dropout(0.5),


    nn.Conv2d(64, 128, kernel_size=(1, 8), padding=(0, 4), bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(),

    nn.Conv2d(128, 256, kernel_size=(1, 4), padding=(0, 2), bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(),


    nn.AdaptiveAvgPool2d((1, 1)),
    
    nn.Flatten(), # (Batch, 256)
    
    # Classifier
    nn.Linear(256, n_classes)  
    
).to(device)

Electrodes (Height): 62, Time (Width): 800


In [14]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)


In [15]:
num_epochs = 20
best_acc=0.0
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for X, y in train_loader:
        X, y = X.to(device).float(), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * X.size(0)
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    train_acc = 100.*correct/total
    train_loss /= total
    
    # Validation / Test
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device).float(), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            test_loss += loss.item() * X.size(0)
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    test_acc = 100.*correct/total
    test_loss /= total
    
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved New Best Model! (New Record: {best_acc:.2f}%)")


Epoch [1/20] Train Loss: 1.2638, Train Acc: 40.92% | Test Loss: 1.4473, Test Acc: 43.61%
Saved New Best Model! (New Record: 43.61%)
Epoch [2/20] Train Loss: 1.1104, Train Acc: 51.33% | Test Loss: 1.0485, Test Acc: 56.54%
Saved New Best Model! (New Record: 56.54%)
Epoch [3/20] Train Loss: 1.0445, Train Acc: 55.09% | Test Loss: 1.1258, Test Acc: 53.81%
Epoch [4/20] Train Loss: 0.9990, Train Acc: 57.62% | Test Loss: 1.0524, Test Acc: 56.51%
Epoch [5/20] Train Loss: 0.9620, Train Acc: 59.72% | Test Loss: 0.9501, Test Acc: 61.13%
Saved New Best Model! (New Record: 61.13%)
Epoch [6/20] Train Loss: 0.9403, Train Acc: 60.73% | Test Loss: 0.9122, Test Acc: 62.89%
Saved New Best Model! (New Record: 62.89%)
Epoch [7/20] Train Loss: 0.9092, Train Acc: 62.11% | Test Loss: 1.0607, Test Acc: 57.90%
Epoch [8/20] Train Loss: 0.8964, Train Acc: 63.18% | Test Loss: 0.9104, Test Acc: 63.83%
Saved New Best Model! (New Record: 63.83%)
Epoch [9/20] Train Loss: 0.8812, Train Acc: 63.89% | Test Loss: 0.9862, T